# 1 Setting Up the Environment

Follow the steps presented at the `README.md` file to install both PyTorch and the Minkowski Engine. And check below if the installation was successful.

In [2]:
import MinkowskiEngine as ME
print(f'MinkowskiEngine version: {ME.__version__}')
import torch
print(f'PyTorch version: {torch.__version__}')

MinkowskiEngine version: 0.5.4
PyTorch version: 2.5.1


Then, we can also check if all of FCGF's extra requirements listed at the `FCGF/requirements.txt` file were properly installed.

In [3]:
%pip install -r ../source/FCGF/requirements.txt

Note: you may need to restart the kernel to use updated packages.


After that, we just need to import everything we are going to use.

In [4]:
import os
import sys
import copy
import numpy as np
import pandas as pd
import open3d as o3d
from urllib.request import urlretrieve
from collections import defaultdict
from datetime import datetime
from zoneinfo import ZoneInfo

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


And set Python to include `source/FCGF` in its search path. Otherwise, since this notebooks is in a different folder, we would not be able to import functions and other things from the FCGF folder.

In [5]:
# Get the absolute path of the source directory
sys.path.append(os.path.abspath("../source/FCGF"))

---

# 2 Input Data

Since we will not retrain the model but use just the pre-trained weights, we can download only the test split. If the train split is ever needed, you can download it too by uncommenting the last block.

In [6]:
test_path = '../data/FCGF/threedmatch_test'
if not os.path.exists(test_path):
  print(f'Downloading data at {test_path}\n\n=================================================================\n')
  !bash ../source/FCGF/scripts/download_3dmatch_test.sh {test_path}
else:
    print(f'The data is already available at {test_path}')


# train_path = '../data/FCGF/threedmatch_train/'
# if not os.path.exists(train_path):
#   print(f'Downloading data at {train_path}\n=================================================================')
#   !bash ../source/FCGF/scripts/download_datasets.sh {train_path}
# else:
#     print(f'The data is already available at {train_path}')

The data is already available at ../data/FCGF/threedmatch_test


Besides that, we also need to download the pre-trained model so we can use it to perform the tests.

In [7]:
# Check if the weight folder has already been created, otherwise creates it
fcgf_weights_folder = '../weigths/FCGF'
if not os.path.exists(fcgf_weights_folder):
    os.makedirs(fcgf_weights_folder)

# Check if the selected pre-trained weights were already downloaded, otherwise download them
fcgf_weight = 'ResUNetBN2C-16feat-3conv.pth'
fcgf_weight_path = os.path.join(fcgf_weights_folder, fcgf_weight)
if not os.path.isfile(fcgf_weight_path):
  print(f'Downloading weight at {fcgf_weight_path}...')
  urlretrieve("https://node1.chrischoy.org/data/publications/fcgf/2019-09-18_14-15-59.pth",
              fcgf_weight_path)
else:
    print(f'Selected weights already available at {fcgf_weight_path}')

Selected weights already available at ../weigths/FCGF/ResUNetBN2C-16feat-3conv.pth


---

# 3 Testing

Here we run the benchmark script to test the FCGF performance in both extraction features itself and providing a registration based on these features. To check the whole implementation of this method as well as details on how this benchmark is performed, please refer to: [github.com/gabriel-corteletti/FCGF](https://github.com/gabriel-corteletti/FCGF).

In [8]:
subset = 3
voxel_size = 0.025
inlier_th = 0.05            # 5 cm --> this must be the same for ICP
model = fcgf_weight_path

run_name = "subset3_fixed_inlier_threshold"

time = datetime.now(ZoneInfo("America/Sao_Paulo")).strftime('%Y-%m-%d_%H-%M-%S')
output_folder = f"../output/FCGF/{run_name}-{time}"
feature_path = f"{output_folder}/features"

!python ../source/FCGF/scripts/benchmark_3dmatch.py \
    --source {test_path} \
    --target {feature_path} \
    --voxel_size {voxel_size} \
    --subset {subset} \
    --model {model} \
    --extract_features \
    --evaluate_feature_match_recall \
    --evaluate_registration

/home/corteletti/miniconda3/envs/fcgf/lib/python3.9/site-packages/MinkowskiEngine/__init__.py:221: UserWarning: The MinkowskiEngine was compiled with CPU_ONLY flag. If you want to compile with CUDA support, make sure `torch.cuda.is_available()` is True when you install MinkowskiEngine.
  warnings.warn(
03/12 04:19:19 ['../data/FCGF/threedmatch_test/7-scenes-redkitchen', '../data/FCGF/threedmatch_test/7-scenes-redkitchen-evaluation', '../data/FCGF/threedmatch_test/sun3d-home_at-home_at_scan1_2013_jan_1', '../data/FCGF/threedmatch_test/sun3d-home_at-home_at_scan1_2013_jan_1-evaluation', '../data/FCGF/threedmatch_test/sun3d-home_md-home_md_scan9_2012_sep_30', '../data/FCGF/threedmatch_test/sun3d-home_md-home_md_scan9_2012_sep_30-evaluation', '../data/FCGF/threedmatch_test/sun3d-hotel_uc-scan3', '../data/FCGF/threedmatch_test/sun3d-hotel_uc-scan3-evaluation', '../data/FCGF/threedmatch_test/sun3d-hotel_umd-maryland_hotel1', '../data/FCGF/threedmatch_test/sun3d-hotel_umd-maryland_hotel1-eval

First we have the **feature extraction** stage, in which the metrics plotted are:
- Average Time.
- FPS (Features Per Second).
- time / feat.: Time necessary to extract one feature.

Then, we have the **feature evaluation** stage, in which the variables plotted are:
- $\tau_1$ **(Feature Distance Threshold)**  
    - Defines the maximum acceptable distance in feature space between the descriptors of a corresponding source/target pair for the putative feature match to be considered an **inlier (true positive)**.  
    - Ensures that feature descriptors are **sufficiently similar**.

- $\tau_2$ **(Inlier Recall Threshold)**  
    - Defines the **minimum acceptable inlier ratio** of a cloud pair (i.e., the percentage of keypoints that are correctly aligned under a **separate geometric distance threshold**).  
    - Ensures whether an alignment is **considered valid**.

- **Feature Match Recall (FMR) Computation**  
    - For each scene, and for the global average across all scenes: $ x.xxxx \pm y.yyyy $ (mean FMR and standard deviation).  
    - FMR computes the fraction of cloud pairs that achieve an **inlier ratio greater than** $ \tau_2 $ out of all possible pairs obtained.
    - The ground-truth transformation is used to **evaluate only the quality of the extracted features**, without considering the final registration step.

To determine whether a feature match and an alignment are successful, the evaluation follows these steps:

1. **Feature Matching:**  
    - Correspondences are found by performing a nearest neighbor search in **feature space**.
    - A match $(x_i, y_i)$ is considered valid if:
    $$ \| f(x_i) - f(y_i) \| \leq \tau_1 $$
    - This ensures that the descriptors are **sufficiently similar**.

2. **Ground-Truth Transformation Application:**  
    - The known ground-truth transformation $ T $ is applied to the source keypoints:
    $$ Tx_i $$

3. **Geometric Inlier Check:**  
    - A match is considered an **inlier** if the transformed source keypoint falls within a **fixed spatial distance threshold** $d_{\text{thresh}}$ (e.g., 10 cm) from the corresponding target keypoint:
    $$ \| Tx_i - y_i \| \leq d_{\text{thresh}} $$
    - This threshold is separate from \( \tau_1 \) and ensures **geometric correctness**.

4. **Inlier Ratio Computation:**  
    - The inlier ratio is computed as:
    $$ \text{Inlier Ratio} = \frac{\text{Number of inliers}}{\text{Total number of feature matches}} $$

5. **Alignment Success Criterion (τ₂):**  
    - If the inlier ratio is **greater than or equal to**  \( \tau_2 \)(e.g., 5%), the alignment is considered **successful**.

After the feature extraction and evaluation, the pairs are matched using **feature-based RANSAC**, a variation of standard RANSAC that incorporates **both geometric and feature similarity information**. The process consists of:
- Obtaining a set of **corresponding points** by performing a **nearest neighbor search in feature space** (matching each source keypoint to the target keypoint with the most similar feature descriptor).
- Running **standard RANSAC** on these correspondences to estimate a rigid transformation while filtering out outliers.

Finally, for **registration evaluation**, the algorithm attempts to align **all non-consecutive point clouds**. The algorithm determines whether a pair was successfully aligned by checking whether the **overlap** between the transformed source cloud and the target cloud exceeds a certain threshold (**30%**). If the alignment is deemed successful, the corresponding transformation is saved to a **log file**.


## 3.1 Quantitative Analysis

Now, we have one `.log` file for each scene. To evaluate the registration performance in terms of **Recall** and **Preccision**, we must use a MATLAB script provided by the author and adapted for our use case (the adapted script is available at: [github.com/gabriel-corteletti/3dmatch-toolbox](github.com/gabriel-corteletti/3dmatch-toolbox)).

Therefore, since we cannot run a MATLAB script in a Python environment, we must open MATLAB and run it from there using the `.log` files we obtained here (stored at `output/FCGF/{run_name}-{time}/registration/logs`)

However, if we are considering just a subset of the test split, we have to adapt also the ground truth files to consider only the results of the same alignments we performed, otherwise the computation of registration recall and precision will be affected by this inconsistency.

To do so, given the subset size we are considering, we first compute all possible pairs to be aligned, i.e. all non-consecutive pairs, and then we create a copy of the original ground truth *.log* and *.info* files where we select only the data related to these possible pairs.

We need to compute all the possible pairs instead of simply considering the same pairs inserted in the obtained registration .log files because it might happen that the algorithm judges itself not able to perform a certain alignment.

In [9]:
def get_matching_pairs(file):
    scene = None
    matching_pairs = defaultdict(list)
    with open(file, 'r') as f:
        for line in f:
            line = line.replace('\n','').split()
            if line[0] == 'Set:':
                scene = line[1]
                continue
            elif scene:
                matching_pairs[scene].append([int(line[0]), int(line[1])])
    return matching_pairs


def get_scene_reducedGT(log_path, info_path, scene_out_gt_path, matching_pairs, scene):

    if not os.path.isdir(scene_out_gt_path):
        os.makedirs(scene_out_gt_path)

    out_log_path = os.path.join(scene_out_gt_path, 'gt.log')
    out_info_path = os.path.join(scene_out_gt_path, 'gt.info')
    copy = False

    n_pairs = len(matching_pairs[scene])
    num_frag = int((1 + np.sqrt(1 + 8*n_pairs))/2)

    with open(log_path, 'r') as i:
        with open(out_log_path, 'w') as o:
            for idx, line in enumerate(i):

                line = line.replace('\n', '').replace('\t', '').split()

                if idx%5 == 0:
                    if [int(line[0]), int(line[1])] in matching_pairs[scene]:
                        o.write(f"{line[0]}\t {line[1]}\t {num_frag}\n")
                        copy = True
                    else:
                        copy = False
                elif copy:
                    o.write(f"{line[0]}\t {line[1]}\t {line[2]}\t {line[3]}\n")

    with open(info_path, 'r') as i:
        with open(out_info_path, 'w') as o:
            for idx, line in enumerate(i):

                line = line.replace('\n', '').replace('\t', '').split()

                if idx%7 == 0:
                    if [int(line[0]), int(line[1])] in matching_pairs[scene]:
                        o.write(f"{line[0]}\t {line[1]}\t {num_frag}\n")
                        copy = True
                    else:
                        copy = False
                elif copy:
                    o.write(f"{line[0]}\t {line[1]}\t {line[2]}\t {line[3]}\t {line[4]}\t {line[5]}\n")


def get_reducedGT(output_folder, test_path):

    out_gt_path = f"{output_folder}/groundtruth"
    if not os.path.isdir(out_gt_path):
        os.makedirs(out_gt_path)

    matching_pairs = get_matching_pairs(f"{output_folder}/registration/matching_pairs.txt")

    for filename in os.listdir(test_path):
        aux = filename.split('-')
        if aux[-1] == 'evaluation':
            log_path = os.path.join(test_path, filename, 'gt.log')
            info_path = os.path.join(test_path, filename, 'gt.info')
            scene_out_gt_path = os.path.join(out_gt_path, filename)
            scene = '-'.join(aux[:-1])
            get_scene_reducedGT(log_path, info_path, scene_out_gt_path, matching_pairs, scene)
    print(f'Reduced ground truth files saved at: {out_gt_path}')

In [10]:
get_reducedGT(output_folder, test_path)

Reduced ground truth files saved at: ../output/FCGF/subset3_fixed_inlier_threshold-2025-03-12_00-19-15/groundtruth


Then we run the script [evaluate.m](https://github.com/gabriel-corteletti/3dmatch-toolbox/blob/master/evaluation/geometric-registration/evaluate.m) to obtain a .csv file containing the registration recall and precision of each scene as well as the overall average results.

After obtaining this, we can upload it back to this environment and present it in a table below.

In [11]:
eval_csv_path = f"{output_folder}/evaluation/registration_evaluation_FCGF.csv"
if os.path.isfile(eval_csv_path):
    reg_eval_FCGF = pd.read_csv(eval_csv_path)
    display(reg_eval_FCGF)
else:
    print(f'No file found at: {eval_csv_path}\nPlease insert the .csv file with the obtained evaluation results in the expected path with the expected file name')

No file found at: ../output/FCGF/subset3_fixed_inlier_threshold-2025-03-12_00-19-15/evaluation/registration_evaluation_FCGF.csv
Please insert the .csv file with the obtained evaluation results in the expected path with the expected file name


We can also compute the fitness and inlier RMSE to compare it to that of ICP. To do so, we need to access the results stored in the output folder of the run. Hence, we define a function responsible for retrieving the obtained transformation for all considered pairs for all scenes, and store it in a DataFrame.

In [12]:
def get_results_from_folder(output_folder, inlier_th):

    results = pd.DataFrame({
    "Scene": pd.Series(dtype='str'),
    "Target": pd.Series(dtype='int'),
    "Source": pd.Series(dtype='int'),
    "Fitness": pd.Series(dtype='float'),
    "Inlier RMSE": pd.Series(dtype='float'),
    "Transformation": pd.Series(dtype='object')
    })
    
    logs_folder = f'{output_folder}/registration/logs'

    for scene in os.listdir(logs_folder):
        scene_name = scene.split('_FCGF')[0]

        found = -1
        transformation = []

        with open(os.path.join(logs_folder, scene), 'r') as f:
            for idx, line in enumerate(f):
                line = line.replace('\n', '').replace('\t', '').split()
                if idx%5 == 0:
                    tgt_ID = int(line[0])
                    src_ID = int(line[1])
                    found = 0
                elif (found > -1) and (found < 4):
                    transformation.append([float(i) for i in line])
                    found += 1
                    if found == 4:


                        source = o3d.io.read_point_cloud(os.path.join(test_path, scene_name, 'cloud_bin_%s.ply' %src_ID))
                        target = o3d.io.read_point_cloud(os.path.join(test_path, scene_name, 'cloud_bin_%s.ply' %tgt_ID))

                        eval = o3d.pipelines.registration.evaluate_registration(source, target, inlier_th, transformation)


                        new_result = pd.DataFrame({'Scene': scene_name,
                                                   'Target': tgt_ID,
                                                   'Source': src_ID,
                                                   'Fitness': eval.fitness,
                                                   'Inlier RMSE': eval.inlier_rmse,
                                                   'Transformation': [transformation]})
                        results = pd.concat([results, new_result], ignore_index=True)
                        found = -1
                        transformation = []

    # save the obtained table as a .csv in the output folder
    filename = f"{output_folder}/registration/registration_table_FCGF.csv"
    results.to_csv(filename, index=False)
    print(f'Registration results table saved at: {filename}')

    return results

In [13]:
results = get_results_from_folder(output_folder, inlier_th)
display(results)

Registration results table saved at: ../output/FCGF/subset3_fixed_inlier_threshold-2025-03-12_00-19-15/registration/registration_table_FCGF.csv


,Scene,Target,Source,Fitness,Inlier RMSE,Transformation
0,sun3d-hotel_umd-maryland_hotel3,33,35,0.507133,0.013664,"[[0.834813522916, 0.012734325044, -0.550385518..."
1,sun3d-hotel_umd-maryland_hotel3,24,28,0.727523,0.013221,"[[0.957323472382, -0.182797103812, 0.223868238..."
2,sun3d-hotel_umd-maryland_hotel3,8,16,0.393190,0.013799,"[[0.929733519016, -0.215941069074, 0.298270076..."
3,sun3d-home_md-home_md_scan9_2012_sep_30,54,56,0.511792,0.016976,"[[0.997886357596, -0.010323114325, -0.06415801..."
4,sun3d-home_md-home_md_scan9_2012_sep_30,8,37,0.946722,0.010027,"[[0.790111486763, -0.437160385104, 0.429668053..."
5,sun3d-hotel_umd-maryland_hotel1,5,13,0.452181,0.016840,"[[0.597849321984, -0.32783931537, 0.7315036373..."
6,sun3d-hotel_umd-maryland_hotel1,15,23,0.599369,0.016188,"[[0.6206611975, -0.312723387679, 0.71901582786..."
7,sun3d-hotel_umd-maryland_hotel1,50,54,0.615390,0.013683,"[[0.924282520011, 0.16311630807, -0.3451012796..."
8,sun3d-hotel_uc-scan3,34,46,0.717001,0.014515,"[[0.650597332371, 0.297081388432, -0.698903254..."
9,7-scenes-redkitchen,48,50,0.956276,0.009676,"[[0.997601542417, 0.007177157276, 0.0688451231..."


In [14]:
def assess_results(results):
    """
    Given a registration result table, computes the mean performance of the alignment
    for all clouds of a specific scene and for that whole split we selected from a dataset.

    Args:
        results (pd.DataFrame): Table containing the registration results of the split

    Returns:
        pd.DataFrame: Table with the overall (mean) performance results
    """

    analysis = results.copy()                                                                           # create a copy of results
    analysis = analysis.drop(["Source", "Target", "Transformation"], axis="columns")              # remove unnecessary columns

    analysis = analysis.groupby(["Scene"]).mean().reset_index()                                         # compute the averages of each scene
    analysis = analysis.rename(columns={"Fitness": "Mean Fitness",                                      # rename columns
                                        "Inlier RMSE": "Mean Inlier RMSE"})

    total_mean = pd.DataFrame({"Scene": "TOTAL", "Mean Fitness": analysis["Mean Fitness"].mean(),       # compute total split average
                               "Mean Inlier RMSE": analysis["Mean Inlier RMSE"].mean()}, index=[0])
    analysis = pd.concat([analysis, total_mean], ignore_index=True)                                     # add total split average

    # ensures the evaluation folder is present
    if not os.path.isdir(f"{output_folder}/evaluation"):
        os.makedirs(f"{output_folder}/evaluation")
    
    # save the obtained registration results summary table as a .csv in the output folder
    filename = f"{output_folder}/evaluation/mean_fitness_and_RMSE_table_FCGF.csv"
    analysis.to_csv(filename, index=False)
    print(f'Registration results summary table saved at: {filename}')

    return analysis

In [15]:
analysis = assess_results(results)
display(analysis)

Registration results summary table saved at: ../output/FCGF/subset3_fixed_inlier_threshold-2025-03-12_00-19-15/evaluation/mean_fitness_and_RMSE_table_FCGF.csv


,Scene,Mean Fitness,Mean Inlier RMSE
0,7-scenes-redkitchen,0.758302,0.015445
1,sun3d-home_at-home_at_scan1_2013_jan_1,0.519403,0.017559
2,sun3d-home_md-home_md_scan9_2012_sep_30,0.729257,0.013501
3,sun3d-hotel_uc-scan3,0.717001,0.014515
4,sun3d-hotel_umd-maryland_hotel1,0.555647,0.015570
5,sun3d-hotel_umd-maryland_hotel3,0.542615,0.013562
6,sun3d-mit_76_studyroom-76-1studyroom2,0.388491,0.019906
7,sun3d-mit_lab_hj-lab_hj_tea_nov_2_2012_scan1_e...,0.558264,0.017125
8,TOTAL,0.596123,0.015898


## 3.2 Qualitative Analysis

We can also perform a qualitative analysis through the visualization of a certain pair with the transformation we obtained. To do so, we use the same visualization function as always plus 

In [16]:
def draw_registration_result(source, target, transformation, voxel_size=0.0):
    """
    Plots the pair of target (cyan) and transformed source (yellow)
    cloud on top of each other. It also performs downsampling, if
    desired, to speed up the visualization.

    Args:
        source (open3d.geometry.PointCloud): Source cloud
        target (open3d.geometry.PointCloud): Target cloud
        transformation (numpy.ndarray): Transformation to be visualized
        voxel_size (float): Resulting size of voxels after downsampling,
                            if desired (default is no downsampling)

    Returns:
        open3d.geometry.PointCloud: Downsampled cloud
        open3d.registration.Feature: Features for registration
    """

    #create copies of both clouds to protect original data
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)

    #downsample for faster visualization (voxel_size is in meters)
    if (voxel_size):
        source_temp = source_temp.voxel_down_sample(voxel_size)
        target_temp = target_temp.voxel_down_sample(voxel_size)

    #paint target cyan and source yellow
    source_temp.paint_uniform_color([1, 0.706, 0])
    target_temp.paint_uniform_color([0, 0.651, 0.929])

    #apply the transformation to the source cloud
    source_temp.transform(transformation)

    #plot target and transformed source clouds
    o3d.visualization.draw_plotly([source_temp, target_temp], width=1200, height=800)

In [17]:
def plot_alignment(output_folder, test_path, inlier_th, scene, tgt_ID, src_ID, voxel_size=0.0, aligned=False):

    
    log_path = f"{output_folder}/registration/logs/{scene}_FCGF.log"

    found = -1
    transformation = []

    print(f"> Scene: {scene}")
    print(f"> Target: {tgt_ID}")
    print(f"> Source: {src_ID}")
    print("------------------------")

    if aligned:
        with open(log_path, 'r') as f:
            for idx, line in enumerate(f):
                line = line.replace('\n', '').replace('\t', '').split()
                if (idx%5 == 0) and (int(line[0]) == tgt_ID) and (int(line[1]) == src_ID):
                    found = 0
                elif (found > -1) and (found < 4):
                    transformation.append([float(i) for i in line])
                    found += 1
                    if found == 4:
                        break
        if transformation == []:
            print('This pair could not be aligned')
            return
        else:
            transformation = np.array(transformation)
            print('AFTER Registration')
    else:
        transformation = np.eye(4)
        print('BEFORE Registration (initial relative pose)')

    source = target = False

    for filename in os.listdir(test_path):
        if filename == scene:

            scene_path = os.path.join(test_path, scene)
            for cloud in os.listdir(scene_path):
                cloud_ID = int(cloud.split('_')[-1].split('.')[0])
                if cloud_ID == src_ID:
                    cloud_path = os.path.join(scene_path, 'cloud_bin_%s.ply' %cloud_ID)
                    source = o3d.io.read_point_cloud(cloud_path)
                elif cloud_ID == tgt_ID:
                    cloud_path = os.path.join(scene_path, 'cloud_bin_%s.ply' %cloud_ID)
                    target = o3d.io.read_point_cloud(cloud_path)

    if source and target:
        eval = o3d.pipelines.registration.evaluate_registration(source, target, inlier_th, transformation)
        print("------------------------")
        print(f"> Fitness: {eval.fitness}")
        print(f"> RMSE: {eval.inlier_rmse}")
        if not (transformation == np.eye(4)).all():
            print(f"> Obtained Transformation:\n{transformation}")
        draw_registration_result(source, target, transformation, voxel_size)
    else:
        print("ERROR: clouds not found")

In [18]:
scenes = ['7-scenes-redkitchen',                                # index: 0
          'sun3d-home_at-home_at_scan1_2013_jan_1',             # index: 1
          'sun3d-home_md-home_md_scan9_2012_sep_30',            # index: 2
          'sun3d-hotel_uc-scan3',                               # index: 3
          'sun3d-hotel_umd-maryland_hotel1',                    # index: 4
          'sun3d-hotel_umd-maryland_hotel3',                    # index: 5
          'sun3d-mit_76_studyroom-76-1studyroom2',              # index: 6
          'sun3d-mit_lab_hj-lab_hj_tea_nov_2_2012_scan1_erika'] # index: 7
plot_alignment(output_folder, test_path, inlier_th,
               scene=scenes[7],
               tgt_ID=14, src_ID=16, voxel_size=0.025, aligned=True)

> Scene: sun3d-mit_lab_hj-lab_hj_tea_nov_2_2012_scan1_erika
> Target: 14
> Source: 16
------------------------
This pair could not be aligned


---